In [1]:
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from pandas import json_normalize
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# --- Configuration ---
DATA_DIR = './data'
# Ensure the data directory exists
os.makedirs(DATA_DIR, exist_ok=True)

# Define file names - USING MULTI-LAYER EMBEDDINGS
USE_MULTILAYER = False  # Set to True to use multi-layer embeddings (2312 features)

if USE_MULTILAYER:
    X_TRAIN_FILE = "X_train_processed_multilayer.npy"
    X_KAGGLE_FILE = "X_kaggle_processed_multilayer.npy"
    print("Using MULTI-LAYER embeddings (layers 6, 9, 12)")
    print("Feature dimensions: 8 structured + 2304 embeddings = 2312 total")
else:
    X_TRAIN_FILE = "X_train_processed.npy"
    X_KAGGLE_FILE = "X_kaggle_processed.npy"
    print("Using SINGLE-LAYER embeddings (layer 12 only)")
    print("Feature dimensions: 8 structured + 768 embeddings = 776 total")

Y_TRAIN_FILE = "y_train.npy"

Using SINGLE-LAYER embeddings (layer 12 only)
Feature dimensions: 8 structured + 768 embeddings = 776 total


In [3]:
# --- 1. Load Data ---
def load_data():
    """Loads the processed feature arrays and the target labels."""
    try:
        X_train_full = np.load(os.path.join(DATA_DIR, X_TRAIN_FILE))
        X_kaggle_full = np.load(os.path.join(DATA_DIR, X_KAGGLE_FILE))
        y_full = np.load(os.path.join(DATA_DIR, Y_TRAIN_FILE))

        print(f"X_train_full loaded: {X_train_full.shape}")
        print(f"X_kaggle_full loaded: {X_kaggle_full.shape}")
        print(f"y_full loaded: {y_full.shape}")

        return X_train_full, X_kaggle_full, y_full

    except FileNotFoundError as e:
        print(f"Error: Could not find required file. Please check DATA_DIR and file names.")
        print(f"Missing file: {e}")
        return None, None, None

X_train_full, X_kaggle_full, y_full = load_data()

if X_train_full is None:
    # Exit if data loading failed
    exit()

# Split the full training data into training and a small validation set
# We use this validation set for a final test after Grid Search
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_full, test_size=0.1, random_state=42, stratify=y_full
)

print("-" * 50)
print(f"Training set size: {X_train.shape[0]}")
print(f"Validation set size: {X_val.shape[0]}")
print("-" * 50)

X_train_full loaded: (154914, 1688)
X_kaggle_full loaded: (103380, 1688)
y_full loaded: (154914,)
--------------------------------------------------
Training set size: 139422
Validation set size: 15492
--------------------------------------------------


In [4]:
from sklearn.base import BaseEstimator, ClassifierMixin
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader, TensorDataset

class PyTorchClassifier(BaseEstimator, ClassifierMixin):
    """Sklearn-compatible wrapper for PyTorch models."""
    
    def __init__(self, model, epochs=10, batch_size=32, lr=0.001, device='cpu'):
        self.model = model
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = lr
        self.device = device
        
    def fit(self, X, y):
        """Train the model - required by sklearn interface."""
        X_tensor = torch.FloatTensor(X).to(self.device)
        y_tensor = torch.LongTensor(y).to(self.device)
        
        dataset = TensorDataset(X_tensor, y_tensor)
        loader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)
        
        self.model.to(self.device)
        criterion = nn.CrossEntropyLoss()
        optimizer = Adam(self.model.parameters(), lr=self.lr)
        
        self.model.train()
        for epoch in range(self.epochs):
            for X_batch, y_batch in loader:
                optimizer.zero_grad()
                outputs = self.model(X_batch)
                loss = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()
        
        return self
    
    def predict(self, X):
        """Make predictions - required by sklearn interface."""
        self.model.eval()
        X_tensor = torch.FloatTensor(X).to(self.device)
        
        with torch.no_grad():
            outputs = self.model(X_tensor)
            predictions = torch.argmax(outputs, dim=1)
        
        return predictions.cpu().numpy()
    
    def predict_proba(self, X):
        """Return probabilities - optional but useful."""
        self.model.eval()
        X_tensor = torch.FloatTensor(X).to(self.device)
        
        with torch.no_grad():
            outputs = self.model(X_tensor)
            probs = torch.softmax(outputs, dim=1)
        
        return probs.cpu().numpy()
    
    def get_params(self, deep=True):
        """Required for GridSearchCV."""
        return {
            'model': self.model,
            'epochs': self.epochs,
            'batch_size': self.batch_size,
            'lr': self.lr,
            'device': self.device
        }
    
    def set_params(self, **params):
        """Required for GridSearchCV."""
        for key, value in params.items():
            setattr(self, key, value)
        return self

In [ ]:
class TweetClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(TweetClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [ ]:
# --- 2. Define Three Model Pipelines for Comparison ---

# We'll compare three different approaches:
# 1. Logistic Regression (baseline)
# 2. MLP (PyTorch Neural Network)
# 3. XGBoost (Gradient Boosting)

pipelines = {
    # 'Logistic Regression': Pipeline([
    #     ('scaler', StandardScaler()),
    #     ('classifier', LogisticRegression(random_state=42, solver='liblinear'))
    # ]),
    
    # 'MLP': Pipeline([
    #     ('scaler', StandardScaler()),
    #     ('classifier', PyTorchClassifier(
    #         model=TweetClassifier(input_dim=776, num_classes=2),
    #         epochs=20,
    #         batch_size=128
    #     ))
    # ]),
    
    'XGBoost': Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', XGBClassifier(
            random_state=42,
            eval_metric='logloss',
            use_label_encoder=False
        ))
    ])
}

print("Created 3 pipelines: Logistic Regression, MLP, and XGBoost")


Created 3 pipelines: Logistic Regression, MLP, and XGBoost


In [9]:
# --- 3. Define Parameter Grids for Each Model ---

# Focused parameter grids for efficient tuning
param_grids = {
    # 'Logistic Regression': {
    #     'classifier__C': [10],
    #     'classifier__penalty': ['l2']
    # },
    
    # 'MLP': {
    #     'classifier__epochs': [20],
    #     'classifier__batch_size': [64],
    #     'classifier__lr': [0.003]
    # },
    
'XGBoost': {
    'classifier__max_depth': [7],  
    'classifier__learning_rate': [0.1],
    'classifier__n_estimators': [500],
    'classifier__subsample': [1.0] 
}

}

print("Parameter grids defined for all three models")

Parameter grids defined for all three models


In [10]:
# --- 4. Run GridSearchCV for All Three Models ---

best_models = {}
cv_scores = {}

for model_name in pipelines.keys():
    print("\n" + "=" * 70)
    print(f"Training: {model_name}")
    print("=" * 70)
    
    # Create GridSearchCV instance
    grid_search = GridSearchCV(
        pipelines[model_name],
        param_grids[model_name],
        cv=3,
        scoring='accuracy',
        n_jobs=-1,
        verbose=1
    )
    
    # Train
    print(f"Starting Grid Search for {model_name}...")
    grid_search.fit(X_train, y_train)
    
    # Store results
    best_models[model_name] = grid_search.best_estimator_
    cv_scores[model_name] = grid_search.best_score_
    
    # Print results
    print(f"\n✅ {model_name} Training Complete")
    print(f"Best CV Accuracy: {grid_search.best_score_:.4f}")
    print(f"Best Parameters: {grid_search.best_params_}")

print("\n" + "=" * 70)
print("All models trained successfully!")
print("=" * 70)


Training: XGBoost
Starting Grid Search for XGBoost...
Fitting 5 folds for each of 1 candidates, totalling 5 fits


/users/eleves-b/2023/khalid.lamrini/.local/lib/python3.9/site-packages/sklearn/utils/extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/users/eleves-b/2023/khalid.lamrini/.local/lib/python3.9/site-packages/sklearn/utils/extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/users/eleves-b/2023/khalid.lamrini/.local/lib/python3.9/site-packages/sklearn/utils/extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/users/eleves-b/2023/khalid.lamrini/.local/lib/python3.9/site-packages/sklearn/utils/extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/users/eleves-b/2023/khalid.lamrini/.local/lib/python3.9/site-packages/sklearn/utils/extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / up


✅ XGBoost Training Complete
Best CV Accuracy: 0.9505
Best Parameters: {'classifier__learning_rate': 0.1, 'classifier__max_depth': 7, 'classifier__n_estimators': 500, 'classifier__subsample': 1.0}

All models trained successfully!


In [11]:
# --- 5. Compare Models on Validation Set ---

val_results = {}

print("\n" + "=" * 70)
print("VALIDATION SET EVALUATION")
print("=" * 70)

for model_name, model in best_models.items():
    # Predict on validation set
    y_val_pred = model.predict(X_val)
    val_acc = accuracy_score(y_val, y_val_pred)
    val_results[model_name] = val_acc
    
    print(f"\n{model_name}:")
    print(f"  CV Score: {cv_scores[model_name]:.4f}")
    print(f"  Val Accuracy: {val_acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_val, y_val_pred))

# Create comparison table
print("\n" + "=" * 70)
print("MODEL COMPARISON SUMMARY")
print("=" * 70)
comparison_df = pd.DataFrame({
    'Model': list(val_results.keys()),
    'CV Accuracy': [cv_scores[m] for m in val_results.keys()],
    'Validation Accuracy': list(val_results.values())
})
comparison_df = comparison_df.sort_values('Validation Accuracy', ascending=False)
print(comparison_df.to_string(index=False))

# Identify best model
best_model_name = max(val_results, key=val_results.get)
best_model = best_models[best_model_name]
print(f"\n🏆 Best Model: {best_model_name} (Val Acc: {val_results[best_model_name]:.4f})")
print("=" * 70)


VALIDATION SET EVALUATION

XGBoost:
  CV Score: 0.9505
  Val Accuracy: 0.9573

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.98      0.96      8268
           1       0.97      0.93      0.95      7224

    accuracy                           0.96     15492
   macro avg       0.96      0.96      0.96     15492
weighted avg       0.96      0.96      0.96     15492


MODEL COMPARISON SUMMARY
  Model  CV Accuracy  Validation Accuracy
XGBoost      0.95051             0.957333

🏆 Best Model: XGBoost (Val Acc: 0.9573)


In [12]:
# --- 6. Generate Final Kaggle Submission with Best Model ---

print("\n" + "=" * 70)
print("GENERATING KAGGLE SUBMISSION")
print("=" * 70)
print(f"Using best model: {best_model_name}")

# Predict on Kaggle test set
print("\nStarting inference on X_kaggle_processed...")
kaggle_predictions = best_model.predict(X_kaggle_full)
print(f"Inference complete. Generated {len(kaggle_predictions)} predictions.")
print(f"Example predictions (first 10): {kaggle_predictions[:10]}")

# Load challenge IDs from test.csv
print("\nBuilding submission_XGBoost.csv...")
X_kaggle = pd.read_json("data/kaggle_test.jsonl", lines=True)
X_kaggle = json_normalize(X_kaggle.to_dict(orient="records"))
challenge_ids = X_kaggle["challenge_id"].values

assert len(challenge_ids) == len(kaggle_predictions), \
    "Mismatch: predictions and challenge_ids lengths differ!"

# Create submission DataFrame
submission = pd.DataFrame({
    "ID": challenge_ids.astype(int),
    "Prediction": kaggle_predictions
})

# Save to CSV
submission.to_csv("./submission/submission_XGBoost.csv", index=False)

print(f"✅ submission.csv created successfully with {len(submission)} predictions!")
print(f"Model used: {best_model_name}")
print("=" * 70)



GENERATING KAGGLE SUBMISSION
Using best model: XGBoost

Starting inference on X_kaggle_processed...
Inference complete. Generated 103380 predictions.
Example predictions (first 10): [1 1 0 1 0 0 0 0 1 0]

Building submission_XGBoost.csv...
✅ submission.csv created successfully with 103380 predictions!
Model used: XGBoost


## Performance Comparison: Single-Layer vs Multi-Layer Embeddings

This cell compares the performance of different embedding approaches:
- **Single-layer (baseline)**: Layer 12 only (768 dims) → 82.6% accuracy
- **Multi-layer (enhanced)**: Layers 6, 9, 12 (2304 dims) → Run to see results
- **Fine-tuning (failed)**: End-to-end training → 70.7% accuracy (overfitting)

**Key Insight**: Multi-layer embeddings capture different levels of linguistic abstraction without the risk of catastrophic forgetting from fine-tuning.
